In [ ]:
!pip uninstall google-generativeai -y
!pip install google-genai

In [ ]:
import os
import json
import random
import time
from google import genai
from google.genai import types
from pydantic import BaseModel


# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create the specific folder and file path
folder_path = "/content/drive/MyDrive/ELARA REPORT"
os.makedirs(folder_path, exist_ok=True)
output_file_path = os.path.join(folder_path, "ELARA_REPORT.jsonl")

# 3. Insert your API Key here
GOOGLE_API_KEY = "your_api_key_here" # <-- PASTE YOUR KEY HERE
client = genai.Client(api_key=GOOGLE_API_KEY)

# 4. Define the exact JSON structure we want Gemini to output
class GenerationOutput(BaseModel):
    session_transcript: str
    elara_report: str

# 5. Configurations for Diversity (Applied Mathematics: Statics & Dynamics)
statics_topics = [
    "الاحتكاك (الاحتكاك على مستوى أفقي خشن، الاحتكاك على مستوى مائل خشن ومعامل الاحتكاك السكوني والحركي)",
    "العزوم (عزم قوة بالنسبة لنقطة في نظام ثنائي الأبعاد، والقياس الجبري للعزم)",
    "العزوم (عزم قوة بالنسبة لنقطة في نظام ثلاثي الأبعاد باستخدام المتجهات)",
    "القوى المتوازية المستوية (محصلة قوتين متوازيتين، اتزان مجموعة من القوى المتوازية)",
    "الاتزان العام (اتزان جسم صلب تحت تأثير مجموعة من القوى المستوية، السلالم والقضبان)",
    "الازدواجات (عزم الازدواج، الازدواج المحصل، تكافؤ واتزان الازدواجات)",
    "مركز الثقل (مركز ثقل جسم متماسك، والكتل السالبة)"
]

dynamics_topics = [
    "تفاضل وتكامل الدوال المتجهة (متجه الموضع، الإزاحة، السرعة، العجلة، والتمثيل البياني لها)",
    "كمية الحركة وقانون نيوتن الأول",
    "قانون نيوتن الثاني (معادلة الحركة، الحركة على مستوى ملس أو خشن، القوة المتغيرة)",
    "تطبيقات قوانين نيوتن (حركة البكرات البسيطة، حركة المصاعد والوزن الظاهري والحقيقي)",
    "الدفع والتصادم (دفع قوة، التصادم المرن وغير المرن، الكرات الملساء)",
    "الشغل (الشغل المبذول من قوة ثابتة، قوة متغيرة، وقوة الوزن)",
    "الطاقة (طاقة الحركة، طاقة الوضع، ومبدأ الشغل والطاقة)",
    "القدرة (القدرة المتوسطة، القدرة اللحظية، وتطبيقات على قدرة المحركات)"
]

student_levels = [
    "Excellent student who makes minor careless mathematical mistakes or sign errors in complex equations.",
    "Average student who memorizes laws but struggles with drawing the free-body diagram and resolving forces.",
    "Weak student with fundamental misconceptions in vectors, forces, or basic calculus, needing step-by-step guidance.",
    "Curious student who asks a lot of 'Why' about the physical meaning of the equations (e.g., 'Why is the normal force not always equal to weight?')."
]

language_styles = [
    "Pure Egyptian Arabic (عامية مصرية بحتة بدون إنجليزي).",
    "Egyptian Arabic mixed heavily with English scientific terms (e.g., 'يا مستر الـ Normal Force دي اتجاهها فين؟').",
    "Standard Arabic with basic English terms (عربي مصري مبسط مع بعض المصطلحات)."
]

# The FIXED System Prompt
QWEN_SYSTEM_PROMPT = "أنت محلل تعليمي خبير (Educational Analyst). مهمتك قراءة سجل المحادثة (Transcript) بين الطالب والمعلم، واستخراج تقرير تحليلي دقيق (ELARA Report) في شكل نقاط يركز على الفجوات المفاهيمية والمفاهيم الخاطئة والعقبات الإدراكية لدى الطالب. لا تذكر أي إحصائيات، ركز فقط على التحليل المعرفي."

def generate_synthetic_data(total_samples=1300, output_file=output_file_path):
    successful_samples = 0
    consecutive_errors = 0 # Track errors for exponential backoff

    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            successful_samples = sum(1 for line in f)

    if successful_samples >= total_samples:
        print(f"File already contains {successful_samples} samples in Drive. Dataset is ready!")
        return

    print(f"Found {successful_samples} saved samples in Google Drive. Resuming generation...")

    while successful_samples < total_samples:
        print(f"Generating sample {successful_samples + 1} of {total_samples}...")

        # Randomly decide between Statics and Dynamics for this session
        subject_branch = random.choice(["Statics (الاستاتيكا)", "Dynamics (الديناميكا)"])

        if "Statics" in subject_branch:
            selected_topic = random.choice(statics_topics)
        else:
            selected_topic = random.choice(dynamics_topics)

        selected_level = random.choice(student_levels)
        selected_lang = random.choice(language_styles)

        prompt = f"""
        You are an expert synthetic data generator. Your task is to generate a highly realistic educational chat session and its corresponding analytical report.

        Context:
        - Subject: Applied Mathematics - {subject_branch} (الرياضيات التطبيقية - الثانوية العامة المصرية)
        - Topic: {selected_topic}
        - Student Profile: {selected_level}
        - Language Style: {selected_lang}

        Instructions:
        1. Generate 'session_transcript':
           - Format MUST strictly follow this pattern with EXPLICIT NEWLINES separating each turn:
             USER: [Student text]
             AI: [Tutor text]
             USER: [Student text]
             AI: [Tutor text]
           - LENGTH REQUIREMENT: The conversation MUST be VERY LONG and detailed. Generate at least 15 to 20 back-and-forth exchanges (USER/AI pairs).
           - CONTENT REQUIREMENT: Since this is Mechanics/Math, involve numerical examples, drawing/imagining free-body diagrams, resolving forces (تحليل القوى), equations of motion, or calculus steps. Let the student struggle with signs (positive/negative directions), angles, or units.
           - It must flow naturally and reflect the assigned student profile and language style.

        2. Generate 'elara_report':
           - This is an analytical report highlighting the student's conceptual gaps and misunderstandings based ONLY on the transcript.
           - FORMAT REQUIREMENT: MUST be formatted STRICTLY as a bulleted list using the (•) symbol. No intro, no outro, no paragraphs. JUST bullet points.
           - LANGUAGE REQUIREMENT: The report MUST be written ENTIRELY in Arabic (اللغة العربية). ONLY use English for scientific terms if it is absolutely necessary and cannot be translated accurately.
        """

        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=GenerationOutput,
                    temperature=0.8,
                    max_output_tokens=8192,
                )
            )

            generated_data = json.loads(response.text.strip())

            qwen_format = {
                "messages": [
                    {"role": "system", "content": QWEN_SYSTEM_PROMPT},
                    {"role": "user", "content": generated_data["session_transcript"]},
                    {"role": "assistant", "content": generated_data["elara_report"]}
                ]
            }

            with open(output_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(qwen_format, ensure_ascii=False) + '\n')

            successful_samples += 1
            consecutive_errors = 0 # Reset error counter on success
            print(f"Sample {successful_samples} generated successfully and synced to Drive! (Branch: {subject_branch})")

            time.sleep(4)

        except Exception as e:
            error_msg = str(e)

            # Check for high demand or rate limit errors
            if any(code in error_msg for code in ["503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"]):
                consecutive_errors += 1
                # Exponential math: 10, 20, 40, 80 seconds + jitter
                wait_time = (10 * (2 ** (consecutive_errors - 1))) + random.uniform(1, 5)
                # Cap the maximum wait time to 5 minutes to avoid indefinite hanging
                wait_time = min(wait_time, 300)

                print(f"API busy or rate limited. Attempt {consecutive_errors} failed. Retrying in {wait_time:.2f} seconds...")
                time.sleep(wait_time)
            else:
                # For unrelated errors, keep the standard short retry so it doesn't break
                print(f"Unexpected error encountered: {error_msg}. Retrying in 10 seconds...")
                time.sleep(10)
            continue

    print(f"\nDataset generation complete. File securely saved at: {output_file}")

# Insert your API key at the top and run
generate_synthetic_data(total_samples=1300)